# Adversarial Training Analysis — Heston Model

Loads all trained networks from `results/adv_nets/` and evaluates:
- **OOS**: 100K held-out Heston paths (seed=42, same distribution as training)
- **OOD**: 100 perturbed Heston configs with ±10% parameter variation

Metrics: **ES₀.₅** and **mean hedging error**

Reproduces He, Sutter & Gonon (2025) Sections 5.2–5.3 Figure 1.

In [ ]:
import sys, math, time
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display
from tqdm.auto import tqdm

sys.path.insert(0, '..')
from src.heston_simulator import HestonParams, HestonSimulator, OutOfSampleHestonSimulator
from src.hedging.hedge_network import HestonHedgeNet
from src.hedging.loss import HestonCVaRLoss

# ── Config — must match src/train_adv_heston.py exactly ─────────────────────
N_SIZES    = [5_000, 10_000, 20_000, 50_000, 100_000]
N_SEEDS    = 3
METHODS    = ['clean', 'S', 'SV']
ALPHA_CVAR = 0.5
K          = 100.0
N_STEPS    = 30
T          = 30 / 365
VP_SCALE   = 1000.0
S0=100.; v0=0.04; kappa=1.; theta=0.04; xi=2.; rho=-0.7
M_TRAIN    = 100_000
M_OOD      = 10_000
N_OOD_CFGS = 100

# Set to 1_000_000 to match paper; 100_000 is faster for interactive use
M_TEST = 100_000

RESULTS_DIR = Path('..') / 'results'
ADV_DIR     = RESULTS_DIR / 'adv_nets'
COLORS = {'clean': 'steelblue', 'S': 'darkorange', 'SV': 'seagreen'}
LABELS = {'clean': 'Clean', 'S': 'S-Attack', 'SV': 'SV-Attack'}

device = (
    torch.device('mps')  if torch.backends.mps.is_available()  else
    torch.device('cuda') if torch.cuda.is_available() else
    torch.device('cpu')
)
print(f'Device: {device}')
print(f'M_TEST={M_TEST:,}  N_OOD_CFGS={N_OOD_CFGS}')
print(f'Looking for networks in: {ADV_DIR.resolve()}')

In [ ]:
# ── Simulate OOS test paths (seed=42 matches training script) ────────────────
print(f'Simulating {M_TEST:,} OOS test paths …')
t0 = time.perf_counter()
test_params = HestonParams(
    S0=S0, v0=v0, kappa=kappa, theta=theta, xi=xi, rho=rho,
    T=T, N=N_STEPS, M=M_TEST,
)
S_te, V_te, VP_te = HestonSimulator(test_params).simulate(seed=42)
VP_te = VP_te * VP_SCALE
print(f'  Done in {time.perf_counter()-t0:.1f}s   shape={S_te.shape}')
print(f'  VP_te mean={VP_te[:,0].mean():.3f}   C_T mean={torch.clamp(S_te[:,-1]-K, min=0.).mean():.4f}')

In [ ]:
# ── Helper functions ─────────────────────────────────────────────────────────

def _make_input(S, V):
    return torch.cat([
        torch.log(S[:, :-1]).unsqueeze(-1),
        V[:, :-1].unsqueeze(-1),
    ], dim=-1)


@torch.no_grad()
def compute_errors(net, S, V, VP, batch_size=10_000):
    """Returns (M,) CPU tensor of per-path hedging errors X = C_T - PnL."""
    net = net.to(device).eval()
    errors = []
    for i in range(0, S.shape[0], batch_size):
        S_b  = S[i:i+batch_size].to(device)
        V_b  = V[i:i+batch_size].to(device)
        VP_b = VP[i:i+batch_size].to(device)
        h    = net(_make_input(S_b, V_b))
        dS   = S_b[:, 1:]  - S_b[:, :-1]
        dVP  = VP_b[:, 1:] - VP_b[:, :-1]
        PnL  = (h[:, :, 0] * dS + h[:, :, 1] * dVP).sum(1)
        X    = torch.clamp(S_b[:, -1] - K, min=0.) - PnL
        errors.append(X.cpu())
    net.cpu()
    return torch.cat(errors)


def es50(errors):
    """ES₀.₅ = mean of the top 50% of hedging errors."""
    k = max(1, int(math.ceil(0.5 * len(errors))))
    return float(torch.topk(errors, k).values.mean())


def mean_err(errors):
    return float(errors.mean())


def load_net(N, method, s):
    p = ADV_DIR / f'N{N}_{method}_s{s}_network.pt'
    if not p.exists():
        return None
    net = HestonHedgeNet(N=N_STEPS, width=20)
    net.load_state_dict(torch.load(p, map_location='cpu', weights_only=True))
    return net


print('Helpers defined.')

In [ ]:
# ── OOS evaluation ────────────────────────────────────────────────────────────
oos_es   = {}   # (N, method, s) → ES50
oos_mean = {}   # (N, method, s) → mean error

all_keys = [
    (N, method, s)
    for N in N_SIZES
    for s in range(min(M_TRAIN // N, N_SEEDS))
    for method in METHODS
]

print(f'Evaluating {len(all_keys)} networks on OOS test set ({M_TEST:,} paths) …\n')
missing = []

for key in tqdm(all_keys, desc='OOS'):
    N, method, s = key
    net = load_net(N, method, s)
    if net is None:
        missing.append(key)
        continue
    errors = compute_errors(net, S_te, V_te, VP_te)
    oos_es  [key] = es50(errors)
    oos_mean[key] = mean_err(errors)

if missing:
    print(f'\nMISSING ({len(missing)}):', [(N, m, s) for N,m,s in missing])

print(f'\nOOS results ({len(oos_es)} networks):')
for N in N_SIZES:
    for method in METHODS:
        n_runs = min(M_TRAIN // N, N_SEEDS)
        vals_es   = [oos_es.get((N,method,s),float('nan')) for s in range(n_runs)]
        vals_mean = [oos_mean.get((N,method,s),float('nan')) for s in range(n_runs)]
        es_str   = ' | '.join(f'{v:.3f}' if not math.isnan(v) else '---' for v in vals_es)
        mean_str = ' | '.join(f'{v:.3f}' if not math.isnan(v) else '---' for v in vals_mean)
        print(f'  N={N:6d} {method:6s}  ES50=[{es_str}]  mean=[{mean_str}]')

In [ ]:
# ── OOD evaluation ────────────────────────────────────────────────────────────
# 100 perturbed Heston configs with ±10% parameter variation, 10K paths each
np.random.seed(0)   # matches training script
base_params = HestonParams(
    S0=S0, v0=v0, kappa=kappa, theta=theta, xi=xi, rho=rho,
    T=T, N=N_STEPS, M=M_TRAIN,
)
ood_sim = OutOfSampleHestonSimulator(base_params, variation=0.1)
ood_es   = {}
ood_mean = {}

found_keys = [k for k in all_keys if k in oos_es]
print(f'OOD evaluation: {len(found_keys)} networks × {N_OOD_CFGS} configs …')
t0 = time.perf_counter()

for key in tqdm(found_keys, desc='OOD nets'):
    N, method, s = key
    net = load_net(N, method, s)
    es_vals, mean_vals = [], []
    for _ in range(N_OOD_CFGS):
        S_o, V_o, VP_o, _ = ood_sim.simulate(M=M_OOD)
        VP_o = VP_o * VP_SCALE
        err = compute_errors(net, S_o, V_o, VP_o)
        es_vals.append(es50(err))
        mean_vals.append(mean_err(err))
    ood_es  [key] = float(np.mean(es_vals))
    ood_mean[key] = float(np.mean(mean_vals))

print(f'OOD done in {(time.perf_counter()-t0)/60:.1f} min')

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for N in N_SIZES:
    for method in METHODS:
        n_runs = min(M_TRAIN // N, N_SEEDS)
        def _agg(d):
            v = [d.get((N,method,s), float('nan')) for s in range(n_runs)]
            v = [x for x in v if not math.isnan(x)]
            return (float(np.mean(v)), float(np.min(v)), float(np.max(v))) if v else (float('nan'),)*3
        oe_m, oe_lo, oe_hi = _agg(oos_es)
        om_m, om_lo, om_hi = _agg(oos_mean)
        de_m, de_lo, de_hi = _agg(ood_es)
        dm_m, dm_lo, dm_hi = _agg(ood_mean)
        rows.append({
            'N': N, 'Method': LABELS[method],
            'OOS ES50':       f'{oe_m:.3f} [{oe_lo:.3f}–{oe_hi:.3f}]',
            'OOS Mean':       f'{om_m:.3f} [{om_lo:.3f}–{om_hi:.3f}]',
            'OOD ES50':       f'{de_m:.3f} [{de_lo:.3f}–{de_hi:.3f}]',
            'OOD Mean':       f'{dm_m:.3f} [{dm_lo:.3f}–{dm_hi:.3f}]',
        })

df = pd.DataFrame(rows).set_index(['N', 'Method'])
print('Summary (mean [min–max] across seeds):')
display(df)

In [ ]:
# ── 4-panel Figure 1 ──────────────────────────────────────────────────────────
# Row 0: ES₀.₅ (OOS | OOD)
# Row 1: Mean hedging error (OOS | OOD)

def _plot_panel(ax, results, title, ylabel):
    for method in METHODS:
        means, mins_, maxs_ = [], [], []
        for N in N_SIZES:
            n_runs = min(M_TRAIN // N, N_SEEDS)
            vals = [results.get((N, method, s), float('nan')) for s in range(n_runs)]
            vals = [v for v in vals if not math.isnan(v)]
            means.append(float(np.mean(vals))  if vals else float('nan'))
            mins_.append(float(np.min(vals))   if vals else float('nan'))
            maxs_.append(float(np.max(vals))   if vals else float('nan'))
        ax.plot(N_SIZES, means, color=COLORS[method], label=LABELS[method],
                linewidth=2.0, marker='o', markersize=4)
        ax.fill_between(N_SIZES, mins_, maxs_, alpha=0.20, color=COLORS[method])
    ax.set_xscale('log')
    ax.set_xlabel('Training Samples N', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, linestyle='--')


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

_plot_panel(axes[0, 0], oos_es,   '(a) Out-of-sample — ES₀.₅',     r'Hedging Loss (ES$_{0.5}$)')
_plot_panel(axes[0, 1], ood_es,   '(b) Out-of-distribution — ES₀.₅',r'Hedging Loss (ES$_{0.5}$)')
_plot_panel(axes[1, 0], oos_mean, '(c) Out-of-sample — Mean Error',  'Mean Hedging Error')
_plot_panel(axes[1, 1], ood_mean, '(d) Out-of-distribution — Mean Error', 'Mean Hedging Error')

fig.suptitle(
    'Heston Adversarial Training — He, Sutter & Gonon (2025)\n'
    'Shaded: min-max range across seeds / partitions',
    fontsize=12, y=1.02,
)
plt.tight_layout()

out_path = RESULTS_DIR / 'heston_figure1_4panel.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved → {out_path}')
plt.show()

In [ ]:
# ── Display the original 2-panel figure saved by the training script ──────────
orig = RESULTS_DIR / 'heston_figure1_adversarial.png'
if orig.exists():
    print('Original figure (ES₀.₅ only, from training script):')
    display(Image(str(orig), width=900))
else:
    print(f'Not found: {orig}')